In [25]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import pickle


In [26]:
columns = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach","exang","oldpeak","slope","ca","thal","num",]

df = pd.read_csv("../data/processed_data/processed.cleveland.data", names=columns, na_values="?")

In [27]:
X = df.dropna().reset_index(drop=True)
y = X["num"]
X.drop(columns=['num'],inplace=True)
X.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal'],
      dtype='object')

In [28]:
y

0      0
1      2
2      1
3      0
4      0
      ..
292    1
293    1
294    2
295    3
296    1
Name: num, Length: 297, dtype: int64

In [29]:
X

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,57.0,0.0,4.0,140.0,241.0,0.0,0.0,123.0,1.0,0.2,2.0,0.0,7.0
293,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0
294,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0
295,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0


In [30]:
y = y.apply(lambda x: 1 if x > 0 else 0)
y

0      0
1      1
2      1
3      0
4      0
      ..
292    1
293    1
294    1
295    1
296    1
Name: num, Length: 297, dtype: int64

In [31]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [32]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cat_cols = ['cp', 'restecg', 'slope','ca', 'thal']

In [33]:
num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
])

In [34]:
num_pipeline_minmax = Pipeline(steps=[
    ('scaler', MinMaxScaler()),
])

In [35]:
cat_pipeline = Pipeline(steps=[
    ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore',sparse_output=False)),
])

In [36]:
preprocessor = ColumnTransformer(transformers=[
    ('num_pipeline', num_pipeline, num_cols),
    ('cat_pipeline', cat_pipeline, cat_cols),
],
    remainder='passthrough',
    n_jobs=-1
)

In [37]:
preprocessor_minmax = ColumnTransformer(transformers=[
    ('num_pipeline_minmax', num_pipeline_minmax, num_cols),
    ('cat_pipeline', cat_pipeline, cat_cols),
],
    remainder='passthrough',
    n_jobs=-1
)

In [38]:
base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42)),
])

In [39]:
pca_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca',PCA(0.95)),
    ('clf',LogisticRegression(max_iter=1000,random_state=42)),
])

In [40]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('rf',RandomForestClassifier(n_estimators=300,random_state=42)),
])



In [41]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [42]:
chi_pipeline = Pipeline([
    ('preprocessor', preprocessor_minmax),
    ('chi2', SelectKBest(score_func=chi2, k=8)),  # select top 8 features
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

In [43]:
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_minmax),
    ('chi2', SelectKBest(score_func=chi2, k=8)),
    ('dt', DecisionTreeClassifier(random_state=42)),
])

In [44]:
svm_pipeline_pca = Pipeline(steps=[
    # ('preprocessor', preprocessor_minmax),
    ('preprocessor', preprocessor),
    # ('chi2', SelectKBest(score_func=chi2, k=8)),
    ("PCA", PCA(0.95)),
    ('svm', SVC(probability=True,random_state=42)),
])

In [45]:
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_minmax),
    ('chi2', SelectKBest(score_func=chi2, k=8)),
    ('svm', SVC(probability=True,random_state=42)),
])

In [46]:
base_pipeline.fit(X_train, y_train)
print("BASE Score: ",base_pipeline.score(X_test, y_test))
print("BASE Classification\n",classification_report(y_test, base_pipeline.predict(X_test)))

pca_pipeline.fit(X_train, y_train)
print("PCA Score: ",pca_pipeline.score(X_test, y_test))
print("PCA Classification\n",classification_report(y_test, pca_pipeline.predict(X_test)))

rf_pipeline.fit(X_train, y_train)
print("RF Score: ",rf_pipeline.score(X_test, y_test))
print("RF Classification\n",classification_report(y_test, rf_pipeline.predict(X_test)))

xgb_pipeline.fit(X_train, y_train)
print("XGB Score: ",xgb_pipeline.score(X_test, y_test))
print("XGB Classification\n",classification_report(y_test, xgb_pipeline.predict(X_test)))

dt_pipeline.fit(X_train, y_train)
print("DT Score: ",dt_pipeline.score(X_test, y_test))
print("DT Classification\n",classification_report(y_test, dt_pipeline.predict(X_test)))

svm_pipeline_pca.fit(X_train, y_train)
print("SVM PCA Score: ",svm_pipeline_pca.score(X_test, y_test))
print("SVM PCA Classification\n",classification_report(y_test, svm_pipeline_pca.predict(X_test)))

svm_pipeline.fit(X_train, y_train)
print("SVM Score: ",svm_pipeline.score(X_test, y_test))
print("SVM Classification\n",classification_report(y_test, svm_pipeline.predict(X_test)))

chi_pipeline.fit(X_train, y_train)
print("CHI Score: ",chi_pipeline.score(X_test, y_test))
print("CHI Classification\n",classification_report(y_test, chi_pipeline.predict(X_test)))

feature_names = preprocessor_minmax.get_feature_names_out()
selected_mask = chi_pipeline.named_steps['chi2'].get_support()
selected_features = feature_names[selected_mask]

print("Selected features:", selected_features)

BASE Score:  0.85
BASE Classification
               precision    recall  f1-score   support

           0       0.91      0.83      0.87        36
           1       0.78      0.88      0.82        24

    accuracy                           0.85        60
   macro avg       0.84      0.85      0.85        60
weighted avg       0.86      0.85      0.85        60

PCA Score:  0.8333333333333334
PCA Classification
               precision    recall  f1-score   support

           0       0.93      0.78      0.85        36
           1       0.73      0.92      0.81        24

    accuracy                           0.83        60
   macro avg       0.83      0.85      0.83        60
weighted avg       0.85      0.83      0.84        60

RF Score:  0.8666666666666667
RF Classification
               precision    recall  f1-score   support

           0       0.91      0.86      0.89        36
           1       0.81      0.88      0.84        24

    accuracy                           0.87

In [47]:

# Dictionary of your pipelines
models = {
    "BASE":base_pipeline,
    "PCA":pca_pipeline,
    "RF":rf_pipeline,
    "XGB":xgb_pipeline,
    "CHI2":chi_pipeline,
    "DT":dt_pipeline,
    "SVM PCA":svm_pipeline_pca,
    "SVM":svm_pipeline,
}

for name, model in models.items():
    print(f"\n===== {name} =====")

    # Get predicted probabilities for ROC/AUC
    try:
        y_probs = model.predict_proba(X_test)[:, 1]  # works for RF, LR, XGB, Chi-Square LR
    except AttributeError:
        # For SVM with probability=False
        y_probs = model.decision_function(X_test)

    # Get predicted labels
    y_pred = model.predict(X_test)

    # Calculate metrics
    roc_auc = roc_auc_score(y_test, y_probs)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Print metrics
    print(f"AUC - ROC Score: {roc_auc:.2f}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")

    # # Plot ROC Curve
    # fpr, tpr, _ = roc_curve(y_test, y_probs)
    # roc_auc_val = auc(fpr, tpr)
    #
    # plt.figure()
    # plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc_val:.2f})')
    # plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    # plt.xlim([0.0, 1.0])
    # plt.ylim([0.0, 1.05])
    # plt.xlabel('False Positive Rate')
    # plt.ylabel('True Positive Rate')
    # plt.title(f'ROC Curve - {name}')
    # plt.legend(loc="lower right")
    # plt.show()


===== BASE =====
AUC - ROC Score: 0.94
Accuracy: 0.85
Precision: 0.78
Recall: 0.88
F1 Score: 0.82

===== PCA =====
AUC - ROC Score: 0.94
Accuracy: 0.83
Precision: 0.73
Recall: 0.92
F1 Score: 0.81

===== RF =====
AUC - ROC Score: 0.96
Accuracy: 0.87
Precision: 0.81
Recall: 0.88
F1 Score: 0.84

===== XGB =====
AUC - ROC Score: 0.90
Accuracy: 0.80
Precision: 0.71
Recall: 0.83
F1 Score: 0.77

===== CHI2 =====
AUC - ROC Score: 0.96
Accuracy: 0.90
Precision: 0.88
Recall: 0.88
F1 Score: 0.88

===== DT =====
AUC - ROC Score: 0.86
Accuracy: 0.90
Precision: 0.91
Recall: 0.83
F1 Score: 0.87

===== SVM PCA =====
AUC - ROC Score: 0.94
Accuracy: 0.88
Precision: 0.81
Recall: 0.92
F1 Score: 0.86

===== SVM =====
AUC - ROC Score: 0.94
Accuracy: 0.90
Precision: 0.91
Recall: 0.83
F1 Score: 0.87


In [ ]:
with open("models/model.pkl", "wb") as f:
    pickle.dump(svm_pipeline_pca, f)

In [ ]:
with open("models/model.pkl", "rb") as f:
    model = pickle.load(f)

age = float(input("Age: "))
sex = float(input("Sex (0 = female, 1 = male): "))
cp = float(input("Chest Pain Type (0–3): "))
trestbps = float(input("Resting Blood Pressure: "))
chol = float(input("Cholesterol: "))
fbs = float(input("Fasting Blood Sugar (0/1): "))
restecg = float(input("Rest ECG (0–2): "))
thalach = float(input("Max Heart Rate: "))
exang = float(input("Exercise Induced Angina (0/1): "))
oldpeak = float(input("Oldpeak: "))
slope = float(input("Slope (0–2): "))
ca = float(input("Number of Major Vessels (0–3): "))
thal = float(input("Thal (0–3): "))

user_data = pd.DataFrame([{
    "age": age,
    "sex": sex,
    "cp": cp,
    "trestbps": trestbps,
    "chol": chol,
    "fbs": fbs,
    "restecg": restecg,
    "thalach": thalach,
    "exang": exang,
    "oldpeak": oldpeak,
    "slope": slope,
    "ca": ca,
    "thal": thal
}])

prediction = model.predict(user_data)[0]
probability = model.predict_proba(user_data).max()

print("\nPrediction:", prediction)
print("Confidence:", round(probability * 100, 2), "%")